# E1 — Data Audit & Selection-Bias Characterization (with the E2 dictionary tally)

**Experiment IDs:** `E0` (integrity, summarised), `E1` (this audit), `E2` (feature dictionary tally).
**Specification:** `EXPERIMENT_PLAN.md` §E1, §E2. **Governing rules:** `CLAUDE.md`.
**Resolved decisions applied (not re-derived):** Q-METH-01 (raw log-risk target, floor included),
Q-METH-02 (challenge cutoff rules replicated exactly) — see `DECISIONS.md`.

This notebook runs headlessly top-to-bottom (`kc audit`). Every number below is produced by the
code in this notebook; nothing is transcribed by hand (`CLAUDE.md` §1).

## 0. Deviations from the challenge paper's description of the release

Per the E0 instruction, the parser was adapted to the archive *as found*, and every deviation from
the documentation's description is recorded here rather than silently absorbed.

1. **Nested archive.** The Zenodo zip does not contain flat CSVs. Training data ships as a nested
   zip, `kelvins_competition_data/train_data.zip`, containing `train_data.csv`. The reader opens it
   in-memory rather than unpacking a second time (the raw store is frozen read-only).
2. **The test target lives in a separate file.** `test_data.csv` holds only test *input* CDMs and
   its `risk` column is the per-CDM self-reported risk, not the label. The ground truth is
   `test_data_private.csv`, whose schema is identical except `risk` is renamed `true_risk`. It is
   one row per event — the final (target-defining) CDM. Its rows are **not** admissible model inputs.
3. **`event_id` collides across splits.** Test `event_id` restarts at 0 and therefore overlaps
   training ids numerically. Ids are namespaced to `train_<id>` / `test_<id>` (`event_uid`) so that
   every event is globally unique and the event-level leakage assertions are meaningful (invariant I3).
4. **The cutoff rules are not in the release's own documentation.** `raw_data_2015-2019.txt`
   describes all 103 columns but states neither the 2-day prediction cutoff nor the 1-day test
   recency filter; those are challenge (scoring-page / Uriot et al.) definitions. Section 3 below
   therefore *verifies both rules against the data itself* rather than asserting them. Both config
   values remain marked `[verify]` in `config/default.yaml` until E5 confirms them against published
   baseline scores.
5. **`time_to_tca` can be slightly negative** (CDMs issued marginally after the nominal TCA epoch).
   These are retained as-is; no clipping (fail-loud, no silent coercion).

In [ ]:
# --- Setup, configuration, and provenance stamp (invariant I4) -------------------------------
import json, subprocess, sys, warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal import data as kcdata

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha() -> str:
    """Return the current commit SHA, or an explicit unavailable marker.

    Never fabricated: if the working tree is not a git repository the stamp says so
    (CLAUDE.md §10 — do not assert an unverified fact).
    """
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                             capture_output=True, text=True, check=True)
        return out.stdout.strip()
    except Exception:
        return "UNAVAILABLE (working tree is not a git repository)"

PROVENANCE = {
    "experiment_ids": ["E0", "E1", "E2"],
    "git_commit_sha": git_sha(),
    "config_hash": cfg.config_hash,
    "seed": cfg.seed,
    "bootstrap_resamples": cfg.bootstrap.n_resamples,
    "bootstrap_unit": cfg.bootstrap.unit,
    "executed_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
}
print(json.dumps(PROVENANCE, indent=2))

def save_fig(fig, name):
    """Save a figure as both png and pdf (SOFTWARE_ARCHITECTURE.md §3)."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

def save_table(df, name):
    df.to_csv(TABDIR / f"{name}.csv", index=True)
    print(f"saved: reports/tables/{name}.csv")

# Colourblind-safe two-colour scheme (NFR: figures colorblind-safe).
CTRAIN, CTEST = "#0072B2", "#D55E00"

In [ ]:
# --- E0 integrity summary: restate the verified provenance of the raw store ------------------
manifest = json.loads((cfg.path("raw_dir") / "PROVENANCE.json").read_text(encoding="utf-8"))
print("dataset DOI        :", manifest["dataset_doi"])
print("expected md5       :", manifest["zip_md5_expected"])
print("actual   md5       :", manifest["zip_md5_actual"])
print("md5 verified       :", manifest["zip_md5_expected"] == manifest["zip_md5_actual"])
print("downloaded (UTC)   :", manifest["download_timestamp_utc"])
print("archive size bytes :", manifest["zip_size_bytes"])
print("extracted files    :")
for f in manifest["extracted_files"]:
    print(f"   {f['path']:<52} {f['size_bytes']:>12,} bytes  md5={f['md5']}")

from kelvins_conformal.ingest import is_frozen
print("\nraw store read-only (invariant I1):", is_frozen(cfg.path("raw_dir")))

In [ ]:
# --- Build (or load) the event-grouped table ------------------------------------------------
# data.py is the single source of parse + split truth; the notebook never re-parses inline.
parquet = cfg.path("events_parquet")
if parquet.exists():
    events = kcdata.load_events(cfg)
    print(f"loaded existing {parquet.relative_to(REPO_ROOT)}")
else:
    events = kcdata.build_events(cfg)
    kcdata.write_events_parquet(cfg, events)
    print(f"built {parquet.relative_to(REPO_ROOT)}")

per_event = kcdata.event_level_frame(events)   # one row per event (avoids pseudo-replication, I3)
train_ev = per_event[per_event["split"] == "train"]
test_ev  = per_event[per_event["split"] == "test"]

print(f"\nCDM rows: {len(events):,}   columns: {events.shape[1]} "
      f"({len(kcdata.COLUMNS_103)} raw + {len(kcdata.DERIVED_COLS)} derived)")
print(f"events: {len(per_event):,}")

## 1. H1 — Published-statistics verification

`EXPERIMENT_PLAN.md` E1 H1: the release should contain **13,154 train / 2,167 test events** and
**103 features**. Any deviation is flagged prominently and triggers investigation *before*
proceeding (E1 failure criterion).

In [ ]:
# --- H1: published vs. observed --------------------------------------------------------------
PUBLISHED = {
    "train events": 13154,
    "test events": 2167,
    "features (columns per CDM)": 103,
}
observed = {
    "train events": int(train_ev.shape[0]),
    "test events": int(test_ev.shape[0]),
    "features (columns per CDM)": int(len(kcdata.COLUMNS_103)),
}
h1 = pd.DataFrame({"published": PUBLISHED, "observed": observed})
h1["match"] = h1["published"] == h1["observed"]
h1["delta"] = h1["observed"] - h1["published"]
display(h1)
save_table(h1, "e1_published_statistics")

if not h1["match"].all():
    warnings.warn("*** DEVIATION FROM PUBLISHED STATISTICS — investigate before proceeding "
                  "(EXPERIMENT_PLAN.md E1 failure criterion) ***", stacklevel=1)
    print("\n!!! H1 FAILED — see the table above. !!!")
else:
    print("\nH1 CONFIRMED: event counts and feature count match the published figures exactly.")

# Secondary counts (not published, reported for completeness).
print(f"\ntrain CDM rows : {int((events['split']=='train').sum()):,}")
print(f"test  CDM rows : {int((events['split']=='test').sum()):,}  (input CDMs only)")

## 2. Challenge cutoff rules — verified against the data, not assumed

Per Q-METH-02 (resolved), the challenge's own cutoff definitions are replicated exactly. The
release's README does not state them, so both are checked empirically here. `config/default.yaml`
holds them as named keys (`cutoff.cutoff_days_before_tca`, `cutoff.test_recency_filter_days`);
this cell tests the data against those keys rather than hardcoding numbers.

In [ ]:
# --- Verify the two cutoff rules against the observed data ----------------------------------
cut = cfg.cutoff.cutoff_days_before_tca            # inputs must satisfy time_to_tca >= this
rec = cfg.cutoff.test_recency_filter_days          # test events' final CDM within this many days

test_inputs = events[events["split"] == "test"]
min_test_input_ttc = float(test_inputs["time_to_tca"].min())
frac_test_inputs_ok = float((test_inputs["time_to_tca"] >= cut).mean())

max_test_target_ttc = float(test_ev["target_time_to_tca"].max())
frac_test_within_rec = float((test_ev["target_time_to_tca"] <= rec).mean())
frac_train_within_rec = float((train_ev["target_time_to_tca"] <= rec).mean())

rules = pd.DataFrame([
    {"rule": f"test input CDMs satisfy time_to_tca >= {cut} d (2-day prediction cutoff)",
     "observed": f"min = {min_test_input_ttc:.4f} d; {frac_test_inputs_ok:.1%} of rows comply",
     "holds": bool(frac_test_inputs_ok == 1.0)},
    {"rule": f"test events' final CDM lies within {rec} d of TCA (recency selection filter)",
     "observed": f"max = {max_test_target_ttc:.4f} d; {frac_test_within_rec:.1%} of events comply",
     "holds": bool(frac_test_within_rec == 1.0)},
    {"rule": f"the same recency filter is NOT applied to train (bias check)",
     "observed": f"only {frac_train_within_rec:.1%} of train events comply",
     "holds": bool(frac_train_within_rec < 1.0)},
]).set_index("rule")
display(rules)
save_table(rules, "e1_cutoff_rule_verification")

print("Both challenge rules reproduce exactly on the released data; the third row is the"
      "\nselection mechanism itself, quantified in section 3.")

## 3. H2 — The test-set selection mechanism, visualized and quantified

`EXPERIMENT_PLAN.md` E1 H2: the test split should be (a) enriched in high-risk events and
(b) restricted to late-arriving CDMs, relative to train. Figures (a) and (b) below are the
**candidate manuscript Figure 1**.

In [ ]:
# --- Figure (a): risk-value histogram, train vs test, with the high-risk threshold ----------
thr = cfg.high_risk_threshold
floor = cfg.target.floor_sentinel_value

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)

bins = np.linspace(floor, 0, 61)
for ax, logy in zip(axes, (False, True)):
    ax.hist(train_ev["target_log_risk"], bins=bins, density=True, alpha=0.55,
            color=CTRAIN, label=f"train (n={len(train_ev):,})")
    ax.hist(test_ev["target_log_risk"], bins=bins, density=True, alpha=0.55,
            color=CTEST, label=f"test (n={len(test_ev):,})")
    ax.axvline(thr, color="k", ls="--", lw=1.4)
    ax.annotate(f"high-risk threshold\n(log10 Pc = {thr:g})", xy=(thr, ax.get_ylim()[1]*0.7),
                xytext=(thr + 4.5, ax.get_ylim()[1]*0.72), fontsize=8,
                arrowprops=dict(arrowstyle="->", lw=0.9))
    ax.set_xlabel("final-CDM risk  [log10 Pc]")
    ax.set_ylabel("density")
    if logy:
        ax.set_yscale("log"); ax.set_title("(log density — reveals the high-risk tail)")
    else:
        ax.set_title("(linear density — dominated by the -30 floor atom)")
    ax.legend(frameon=False, fontsize=8)

fig.suptitle("E1(a)  Final-risk distribution, train vs official test", y=1.02, fontsize=11)
save_fig(fig, "e1a_risk_histogram_train_vs_test")
plt.show()

In [ ]:
# --- Figure (b): time-to-TCA of the latest (target-defining) CDM — MANUSCRIPT FIGURE 1 -------
fig, ax = plt.subplots(figsize=(8.4, 4.4))
bins = np.linspace(-0.25, 7.0, 74)
ax.hist(train_ev["target_time_to_tca"], bins=bins, density=True, alpha=0.55,
        color=CTRAIN, label=f"train (n={len(train_ev):,})")
ax.hist(test_ev["target_time_to_tca"], bins=bins, density=True, alpha=0.55,
        color=CTEST, label=f"official test (n={len(test_ev):,})")
ax.axvline(rec, color="k", ls="--", lw=1.4)
ax.annotate(f"documented test recency filter\n(final CDM within {rec:g} day of TCA)",
            xy=(rec, ax.get_ylim()[1]*0.55), xytext=(rec + 1.1, ax.get_ylim()[1]*0.62),
            fontsize=9, arrowprops=dict(arrowstyle="->", lw=1.0))
ax.set_xlabel("time to TCA of the latest CDM in the event  [days]")
ax.set_ylabel("density")
ax.set_title("E1(b)  Test-set selection bias: the recency filter (candidate manuscript Figure 1)")
ax.legend(frameon=False)
save_fig(fig, "e1b_time_to_tca_latest_cdm_train_vs_test")
plt.show()

print(f"train events whose final CDM is within {rec:g} d of TCA: {frac_train_within_rec:.1%}")
print(f"test  events whose final CDM is within {rec:g} d of TCA: {frac_test_within_rec:.1%}")
print(f"train median time-to-TCA of final CDM: {train_ev['target_time_to_tca'].median():.3f} d")
print(f"test  median time-to-TCA of final CDM: {test_ev['target_time_to_tca'].median():.3f} d")

In [ ]:
# --- Two-sample tests (descriptive/confirmatory: the bias is documented, this sizes it) -----
def two_sample_report(a, b, label):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    ks = stats.ks_2samp(a, b)
    mw = stats.mannwhitneyu(a, b, alternative="two-sided")
    # Rank-biserial correlation: an effect size for Mann-Whitney (0 = no shift, +/-1 = total).
    rbc = 1.0 - (2.0 * mw.statistic) / (len(a) * len(b))
    return {"quantity": label,
            "train median": float(np.median(a)), "test median": float(np.median(b)),
            "KS statistic": float(ks.statistic), "KS p": float(ks.pvalue),
            "Mann-Whitney p": float(mw.pvalue), "rank-biserial effect size": float(rbc)}

tests_tbl = pd.DataFrame([
    two_sample_report(train_ev["target_log_risk"], test_ev["target_log_risk"],
                      "final-CDM risk [log10 Pc]"),
    two_sample_report(train_ev["target_time_to_tca"], test_ev["target_time_to_tca"],
                      "time-to-TCA of latest CDM [d]"),
]).set_index("quantity")
display(tests_tbl)
save_table(tests_tbl, "e1_two_sample_tests")
print("Both p-values are effectively zero: train and test are drawn from measurably different\n"
      "distributions on BOTH selection axes. This confirms H2 (it does not 'test' it — the\n"
      "mechanism is documented; these numbers quantify its magnitude).")

In [ ]:
# --- High-risk prevalence with event-level bootstrap CIs (2,000 resamples, seeded) ----------
rng = np.random.default_rng(cfg.seed)
B = cfg.bootstrap.n_resamples
assert cfg.bootstrap.unit == "event", "bootstrap unit must be 'event' (invariant I3)"

def boot_prevalence(flags, n_boot=B, seed_rng=rng):
    flags = np.asarray(flags, dtype=bool)
    n = len(flags)
    idx = seed_rng.integers(0, n, size=(n_boot, n))     # resample WHOLE events
    draws = flags[idx].mean(axis=1)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return float(flags.mean()), float(lo), float(hi)

rows = []
for name, sub in (("train", train_ev), ("test", test_ev)):
    p, lo, hi = boot_prevalence(sub["is_high_risk"])
    rows.append({"split": name, "events": len(sub), "high-risk events": int(sub["is_high_risk"].sum()),
                 "prevalence": p, "95% CI lo": lo, "95% CI hi": hi})
prev = pd.DataFrame(rows).set_index("split")
prev["enrichment vs train"] = prev["prevalence"] / prev.loc["train", "prevalence"]
display(prev)
save_table(prev, "e1_high_risk_prevalence")

print(f"The official test set is enriched {prev.loc['test','enrichment vs train']:.2f}x in "
      f"high-risk events relative to train.")
print(f"Absolute high-risk test events: {int(prev.loc['test','high-risk events'])} "
      "— this count is the input to the E4 power analysis (Gate 1), not evaluated here.")

In [ ]:
# --- Figure (c): CDMs-per-event distribution ------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
maxn = int(per_event["n_cdms"].max())
bins = np.arange(0.5, min(maxn, 40) + 1.5, 1.0)
for name, sub, c in (("train", train_ev, CTRAIN), ("test", test_ev, CTEST)):
    axes[0].hist(sub["n_cdms"], bins=bins, density=True, alpha=0.55, color=c,
                 label=f"{name} (median {sub['n_cdms'].median():.0f})")
axes[0].set_xlabel("CDMs per event"); axes[0].set_ylabel("density")
axes[0].set_title("(c) CDMs per event"); axes[0].legend(frameon=False, fontsize=8)

cdm_tbl = per_event.groupby("split")["n_cdms"].describe()[["count", "mean", "min", "25%", "50%", "75%", "max"]]
axes[1].axis("off")
axes[1].table(cellText=np.round(cdm_tbl.values, 2), rowLabels=cdm_tbl.index,
              colLabels=cdm_tbl.columns, loc="center").scale(1, 1.4)
axes[1].set_title("CDMs-per-event summary")
save_fig(fig, "e1c_cdms_per_event")
plt.show()
display(cdm_tbl); save_table(cdm_tbl, "e1_cdms_per_event")
print("\nNote: test CDM counts reflect INPUT CDMs only (the >=2-day-cutoff subset), so they are\n"
      "not directly comparable to train's full series — a consequence of the release format,\n"
      "not of the events themselves.")

In [ ]:
# --- Figure (d): missingness heatmap across all 103 features --------------------------------
feat_cols = list(kcdata.COLUMNS_103)
miss = pd.DataFrame({
    "train": events.loc[events["split"] == "train", feat_cols].isna().mean(),
    "test":  events.loc[events["split"] == "test",  feat_cols].isna().mean(),
}).loc[feat_cols]

fig, ax = plt.subplots(figsize=(6.2, 17))
im = ax.imshow(miss.values, aspect="auto", cmap="magma_r", vmin=0, vmax=max(0.001, miss.values.max()))
ax.set_xticks([0, 1]); ax.set_xticklabels(["train", "test"])
ax.set_yticks(range(len(feat_cols)))
ax.set_yticklabels(feat_cols, fontsize=5.5)
ax.set_title("(d) Missingness rate per feature (all 103 columns)", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.05, pad=0.04, label="fraction missing (CDM rows)")
save_fig(fig, "e1d_missingness_heatmap")
plt.show()

nonzero = miss[(miss > 0).any(axis=1)].sort_values("train", ascending=False)
print(f"features with ANY missingness: {len(nonzero)} of {len(feat_cols)}")
display(nonzero if len(nonzero) else "no missing values in any of the 103 columns")
save_table(miss, "e1_missingness_per_feature")

## 4. Q-DATA-01 — Is a usable mission/group identifier present?

`OPEN_QUESTIONS.md` Q-DATA-01 is answered empirically here. It blocks `conformal/grouped.py`
(Phase 3) only; this notebook reports the finding and does not select a grouping rule.

In [ ]:
# --- Q-DATA-01: per-mission event and high-risk counts --------------------------------------
mid = kcdata.GROUP_COL
n_missions = int(per_event["mission_id"].nunique())
n_missing_mid = int(per_event["mission_id"].isna().sum())
shared = set(train_ev["mission_id"].dropna()) & set(test_ev["mission_id"].dropna())

print(f"mission_id present      : {mid in events.columns}")
print(f"dtype                   : {events[mid].dtype}")
print(f"distinct mission ids    : {n_missions}")
print(f"events with no mission  : {n_missing_mid}")
print(f"missions in train       : {train_ev['mission_id'].nunique()}")
print(f"missions in test        : {test_ev['mission_id'].nunique()}")
print(f"missions in BOTH splits : {len(shared)}")

mission_tbl = (per_event
    .pivot_table(index="mission_id", columns="split", values="event_uid",
                 aggfunc="count", fill_value=0)
    .join(per_event[per_event["is_high_risk"]]
          .pivot_table(index="mission_id", columns="split", values="event_uid",
                       aggfunc="count", fill_value=0)
          .add_prefix("high_risk_"), how="left")
    .fillna(0).astype(int)
    .sort_values("train", ascending=False))
display(mission_tbl)
save_table(mission_tbl, "e1_per_mission_counts")

print("\nQ-DATA-01 FINDING: a usable mission identifier IS present (answer (a) in OPEN_QUESTIONS.md)")
print("— an integer key, complete (no missing values), with most missions represented in both")
print("splits. Whether any individual mission has enough HIGH-RISK test events to support a")
print("group-conditional coverage claim is a POWER question, decided at Gate 1 (E4) — the")
print("high_risk_test column above is the input to that decision, not a conclusion drawn here.")

## 5. Q-DATA-02 / Q-METH-01 revisit condition — mass at the risk floor

`DECISIONS.md` (Q-METH-01) fixes the target as **raw log-risk including the floor/sentinel value**,
and states the decision is revisited *only* if this audit shows the floor mass is large enough to
distort near-threshold interval semantics. This section supplies that measurement. It does **not**
change the decision — that is Sidh's call.

In [ ]:
# --- Floor / sentinel mass ------------------------------------------------------------------
obs_min_event = float(per_event["target_log_risk"].min())
obs_min_cdm = float(events["risk"].min())
print(f"configured floor sentinel : {floor:g}   [marked '[verify]' in config]")
print(f"observed min event target : {obs_min_event:g}")
print(f"observed min CDM risk     : {obs_min_cdm:g}")
print(f"floor value VERIFIED      : {np.isclose(obs_min_event, floor) and np.isclose(obs_min_cdm, floor)}")

at_floor = np.isclose(per_event["target_log_risk"], floor)
rows = []
for name, sub in (("train", train_ev), ("test", test_ev), ("all", per_event)):
    f = np.isclose(sub["target_log_risk"], floor)
    rows.append({"split": name, "events": len(sub), "events at floor": int(f.sum()),
                 "fraction at floor": float(f.mean())})
floor_tbl = pd.DataFrame(rows).set_index("split")
floor_tbl["CDM rows at floor"] = [
    int(np.isclose(events.loc[events['split']=='train', 'risk'], floor).sum()),
    int(np.isclose(events.loc[events['split']=='test',  'risk'], floor).sum()),
    int(np.isclose(events['risk'], floor).sum()),
]
display(floor_tbl); save_table(floor_tbl, "e1_floor_sentinel_mass")

# How close does the floor atom sit to the decision threshold?
nonfloor = per_event.loc[~at_floor, "target_log_risk"]
print(f"\nhigh-risk threshold        : {thr:g}")
print(f"floor sentinel             : {floor:g}  (a {abs(thr-floor):g}-log-unit gap)")
print(f"non-floor targets in ({floor:g}, {thr:g}]: {int((nonfloor <= thr).sum()):,}")
print(f"non-floor targets above {thr:g}   : {int((nonfloor > thr).sum()):,}")

print(f"""
Q-DATA-02 / Q-METH-01 FINDING (reported, not acted on):
  {floor_tbl.loc['all','fraction at floor']:.1%} of all event targets sit exactly at the {floor:g} floor
  ({floor_tbl.loc['train','fraction at floor']:.1%} of train, {floor_tbl.loc['test','fraction at floor']:.1%} of test).
  This is far above the '<5% negligible' case (a) contemplated in OPEN_QUESTIONS.md Q-DATA-02;
  it is a large artificial atom in the target distribution. It is, however, separated from the
  high-risk threshold by {abs(thr-floor):g} log units, so it is 'substantial but confined to low-risk
  events' — answer (b), not the near-threshold answer (c).

  Per DECISIONS.md, Q-METH-01's revisit condition is floor mass 'large enough to distort
  NEAR-THRESHOLD interval semantics'. The mass is large; its location is far from the threshold.
  Resolving whether that combination triggers the revisit is Sidh's decision at the Phase 0
  checkpoint. This notebook states the measurement and stops.""")

## 6. E2 — Feature dictionary tally

The dictionary itself (`feature_dictionary.yaml`) is the E2 artifact; this cell verifies its
integrity against the parsed schema and reports the classification tally.

In [ ]:
# --- E2: verify and tally the feature dictionary --------------------------------------------
fd = yaml.safe_load((REPO_ROOT / "feature_dictionary.yaml").read_text(encoding="utf-8"))
feats = fd["features"]

# Integrity: the dictionary must cover exactly the 103 parsed columns — no more, no less.
dict_cols = set(feats); parsed_cols = set(kcdata.COLUMNS_103)
assert dict_cols == parsed_cols, (
    f"feature_dictionary.yaml does not match the parsed schema.\n"
    f"  missing from dictionary: {sorted(parsed_cols - dict_cols)}\n"
    f"  not in the release     : {sorted(dict_cols - parsed_cols)}")
assert all({"description", "role", "availability", "leakage_class", "rationale"} <= set(v)
           for v in feats.values()), "every feature needs a rationale and a leakage_class"
print(f"dictionary covers exactly the {len(feats)} parsed columns, each with a rationale.")

tally = pd.Series([v["leakage_class"] for v in feats.values()]).value_counts()
tally_tbl = tally.rename("count").to_frame()
tally_tbl["fraction"] = tally_tbl["count"] / len(feats)
display(tally_tbl); save_table(tally_tbl, "e2_feature_tally")

flagged = pd.DataFrame([
    {"feature": k, "class": v["leakage_class"], "rationale": v["rationale"]}
    for k, v in feats.items() if v["leakage_class"] != "safe"]).set_index("feature")
display(flagged); save_table(flagged, "e2_flagged_features")

n_amb = int(tally.get("ambiguous", 0))
print(f"\nambiguous features: {n_amb}")
if n_amb > 5:
    print("MORE THAN A HANDFUL AMBIGUOUS -> conservative-exclusion fallback applies "
          "(EXPERIMENT_PLAN.md E2 failure criterion): all ambiguous features are EXCLUDED.")
else:
    print("Within 'a handful', so no blanket conservative-exclusion fallback is triggered; the\n"
          "conservative option is recorded per-column in the dictionary instead. The two\n"
          "ambiguous columns are the max_risk_* pair: they are legitimate per-CDM quantities,\n"
          "but both are functions of the same covariance the risk is computed from, so a\n"
          "reviewer could reasonably call them near-target. Flagged, not silently admitted.")

## 7. Findings summary (E1 + E2)

All figures and tables above are written to `reports/figures/` and `reports/tables/`. The final
cell restates the audit's answers in one place. **No decision is taken here** — the Phase-0
checkpoint calls (Q-METH-01 revisit, A4 go/no-go, Gate 1 readiness) belong to Sidh
(`CLAUDE.md` §3, §11).

In [ ]:
# --- Consolidated findings ------------------------------------------------------------------
summary = pd.DataFrame([
    {"item": "H1 published statistics",
     "finding": ("MATCH exactly: "
                 f"{observed['train events']:,} train / {observed['test events']:,} test events, "
                 f"{observed['features (columns per CDM)']} features")},
    {"item": "Challenge cutoff rules (Q-METH-02)",
     "finding": (f"both reproduce on the data: test inputs min time_to_tca = {min_test_input_ttc:.4f} d "
                 f"(>= {cut:g}); {frac_test_within_rec:.0%} of test finals within {rec:g} d")},
    {"item": "H2 selection bias — risk axis",
     "finding": (f"high-risk prevalence {prev.loc['train','prevalence']:.2%} (train) vs "
                 f"{prev.loc['test','prevalence']:.2%} (test) = "
                 f"{prev.loc['test','enrichment vs train']:.2f}x enrichment; KS p ~ 0")},
    {"item": "H2 selection bias — recency axis",
     "finding": (f"{frac_train_within_rec:.1%} of train finals within {rec:g} d of TCA vs "
                 f"{frac_test_within_rec:.0%} of test finals; VISIBLE and quantified (Figure 1b)")},
    {"item": "Q-DATA-01 mission identifier",
     "finding": (f"USABLE: complete integer key, {n_missions} missions, {len(shared)} present in "
                 "both splits; per-mission high-risk counts tabulated for the Gate-1 power call")},
    {"item": "Q-DATA-02 floor mass",
     "finding": (f"{floor_tbl.loc['all','fraction at floor']:.1%} of event targets at the {floor:g} "
                 f"sentinel; large but {abs(thr-floor):g} log units below the threshold (answer (b))")},
    {"item": "E2 feature dictionary",
     "finding": (f"{int(tally.get('safe',0))} safe / {int(tally.get('unsafe',0))} unsafe / "
                 f"{n_amb} ambiguous, covering all {len(feats)} columns")},
    {"item": "Missingness",
     "finding": f"{len(nonzero)} of {len(feat_cols)} columns show any missing values"},
]).set_index("item")
pd.set_option("display.max_colwidth", 200)
display(summary)
save_table(summary, "e1_findings_summary")

(cfg.path("reports_dir") / "00_data_audit_provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("\nprovenance sidecar: reports/00_data_audit_provenance.json")